# DuckDB Fundamentals Explorer

Notebook for validating V2 fundamentals storage after the dual-layer update:
- raw CAS blobs in `data/fundamentals/blobs/`
- endpoint analytics parquet in `data/fundamentals/parquet/endpoint=...`
- normalized analytics parquet in `data/fundamentals/parquet/analytics=...`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

ROOT = Path('/Users/alex/Documents/pystocks')
DUCKDB_PATH = ROOT / 'data' / 'fundamentals' / 'fundamentals.duckdb'
PARQUET_ROOT = ROOT / 'data' / 'fundamentals' / 'parquet'
assert DUCKDB_PATH.exists(), f'Missing DuckDB file: {DUCKDB_PATH}'

con = duckdb.connect(str(DUCKDB_PATH))
print('Connected:', DUCKDB_PATH)
print('Parquet root exists:', PARQUET_ROOT.exists())

def has_relation(name: str) -> bool:
    row = con.execute(
        "SELECT 1 FROM information_schema.tables WHERE lower(table_name) = lower(?) LIMIT 1",
        [name],
    ).fetchone()
    return row is not None


def has_column(relation: str, column: str) -> bool:
    row = con.execute(
        "SELECT 1 FROM information_schema.columns WHERE lower(table_name) = lower(?) AND lower(column_name) = lower(?) LIMIT 1",
        [relation, column],
    ).fetchone()
    return row is not None

def safe_df(query: str, requires=None) -> pd.DataFrame:
    requires = requires or []
    missing = [name for name in requires if not has_relation(name)]
    if missing:
        return pd.DataFrame({'missing_relation': missing})
    return con.sql(query).df()


## 1) Objects and view inventory

In [ ]:
print(safe_df("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
"""))

## 2) Endpoint + analytics catalog coverage

In [ ]:
endpoint_catalog = safe_df("SELECT * FROM endpoint_catalog ORDER BY endpoint", requires=['endpoint_catalog'])
analytics_catalog = safe_df("SELECT * FROM analytics_catalog ORDER BY analytics_name", requires=['analytics_catalog'])

print('Endpoint catalog rows:', len(endpoint_catalog))
print('Analytics catalog rows:', len(analytics_catalog))

endpoint_catalog

In [ ]:
analytics_catalog

## 3) Recent endpoint events (metadata)

In [ ]:
recent_events = safe_df("""
SELECT
  conid,
  endpoint,
  effective_at,
  observed_at,
  effective_source,
  payload_hash,
  payload_size_raw,
  payload_size_compressed
FROM endpoint_events_all
ORDER BY observed_at DESC
""", requires=['endpoint_events_all'])
recent_events

## 4) Complex endpoint snapshots (`endpoint_dividends`, `endpoint_ownership`)

In [ ]:
dividends_snapshot = safe_df("""
SELECT
  conid,
  effective_at,
  a__response_type,
  a__has_history,
  a__history_points,
  a__embedded_price_points,
  a__no_div_data_marker,
  a__dividend_yield_ttm,
  a__dividend_ttm,
  a__last_paid_date,
  a__last_paid_amount,
  a__last_paid_currency
FROM endpoint_dividends
ORDER BY effective_at DESC, conid
LIMIT 200
""", requires=['endpoint_dividends'])
dividends_snapshot

In [ ]:
ownership_snapshot = safe_df("""
SELECT
  conid,
  effective_at,
  a__owners_types_count,
  a__institutional_owners_count,
  a__insider_owners_count,
  a__trade_log_count_raw,
  a__trade_log_count_kept,
  a__ownership_history_price_points,
  a__institutional_total_pct,
  a__insider_total_pct
FROM endpoint_ownership
ORDER BY effective_at DESC, conid
LIMIT 200
""", requires=['endpoint_ownership'])
ownership_snapshot

## 5) Normalized analytics datasets

In [ ]:
analytics_counts = safe_df("""
SELECT analytics_name, COUNT(*) AS n_rows, COUNT(DISTINCT conid) AS n_conids
FROM analytics_rows_all
GROUP BY analytics_name
ORDER BY analytics_name
""", requires=['analytics_rows_all'])
analytics_counts

In [ ]:
safe_df("""
SELECT
  conid, event_date, amount, currency, description, declaration_date, record_date, payment_date, effective_at
FROM analytics_dividends_events
ORDER BY event_date DESC, conid
LIMIT 200
""", requires=['analytics_dividends_events'])

In [ ]:
safe_df("""
SELECT
  conid, metric_id, value, formatted_value, effective_at
FROM analytics_dividends_industry_metrics
ORDER BY effective_at DESC, conid, metric_id
LIMIT 200
""", requires=['analytics_dividends_industry_metrics'])

In [ ]:
safe_df("""
SELECT *
FROM analytics_ownership_trade_log
LIMIT 200
""", requires=['analytics_ownership_trade_log'])

## 6) Integrity checks

In [ ]:
if has_column('analytics_ownership_trade_log', 'action'):
    no_change_check = safe_df("""
SELECT COUNT(*) AS no_change_rows
FROM analytics_ownership_trade_log
WHERE upper(action) = 'NO CHANGE'
""", requires=['analytics_ownership_trade_log'])
else:
    no_change_check = pd.DataFrame([
        {
            'no_change_rows': 0,
            'note': 'action column absent (likely no ownership_trade_log rows yet)'
        }
    ])

lineage_check = safe_df("""
SELECT
  COUNT(*) AS analytics_rows,
  SUM(CASE WHEN e.event_id IS NULL THEN 1 ELSE 0 END) AS missing_event_join,
  SUM(CASE WHEN e.payload_hash IS DISTINCT FROM a.payload_hash THEN 1 ELSE 0 END) AS payload_hash_mismatch
FROM analytics_rows_all a
LEFT JOIN endpoint_events_all e
  ON a.endpoint_event_id = e.event_id
""", requires=['analytics_rows_all', 'endpoint_events_all'])

print('Ownership NO CHANGE rows (expected 0):')
print(no_change_check)
print('Lineage integrity (expected missing_event_join=0, payload_hash_mismatch=0):')
lineage_check


## 7) Parquet partition checks (filesystem + direct read)

In [ ]:
def count_parquet_files(glob_expr: str) -> int:
    return len(list(PARQUET_ROOT.glob(glob_expr)))

parquet_inventory = pd.DataFrame([
    {'group': 'endpoint', 'name': 'all', 'files': count_parquet_files('endpoint=*/year=*/month=*/*.parquet')},
    {'group': 'endpoint', 'name': 'dividends', 'files': count_parquet_files('endpoint=dividends/year=*/month=*/*.parquet')},
    {'group': 'endpoint', 'name': 'ownership', 'files': count_parquet_files('endpoint=ownership/year=*/month=*/*.parquet')},
    {'group': 'analytics', 'name': 'dividends_events', 'files': count_parquet_files('analytics=dividends_events/year=*/month=*/*.parquet')},
    {'group': 'analytics', 'name': 'dividends_industry_metrics', 'files': count_parquet_files('analytics=dividends_industry_metrics/year=*/month=*/*.parquet')},
    {'group': 'analytics', 'name': 'ownership_trade_log', 'files': count_parquet_files('analytics=ownership_trade_log/year=*/month=*/*.parquet')},
])
parquet_inventory

In [ ]:
endpoint_glob = str(PARQUET_ROOT / 'endpoint=*/year=*/month=*/*.parquet')
analytics_glob = str(PARQUET_ROOT / 'analytics=*/year=*/month=*/*.parquet')

endpoint_direct = con.sql(f"""
SELECT endpoint, COUNT(*) AS n_rows
FROM read_parquet('{endpoint_glob}', union_by_name=true)
GROUP BY endpoint
ORDER BY endpoint
""").df()

if count_parquet_files('analytics=*/year=*/month=*/*.parquet') > 0:
    analytics_direct = con.sql(f"""
    SELECT analytics_name, COUNT(*) AS n_rows
    FROM read_parquet('{analytics_glob}', union_by_name=true)
    GROUP BY analytics_name
    ORDER BY analytics_name
    """).df()
else:
    analytics_direct = pd.DataFrame(columns=['analytics_name', 'n_rows'])

print('Endpoint parquet direct counts:')
display(endpoint_direct)
print('Analytics parquet direct counts:')
analytics_direct

In [ ]:
# Optional cleanup when done with notebook session
con.close()